# Gathering SCAI Data

## 1. Stroke Centers per 100k

## Getting Population Data

In [1]:
import pandas as pd

In [2]:
import requests

url = "https://api.census.gov/data/2023/acs/acs1"

API_KEY = "3d1a8efb010ab94526aed9bb9b1b8ba58c722e3d"

states = ["09", "34", "36"]  # CT, NJ, NY

all_data = []

for s in states:
    params = {
        "get": "NAME,B01003_001E",
        "for": "county:*",
        "in": f"state:{s}",
        "key": API_KEY
    }

    r = requests.get(url, params=params)

    print("\nSTATE:", s)
    print("STATUS:", r.status_code)
    print("CONTENT (first 300 chars):")
    print(r.text[:300])
    print("-" * 50)


STATE: 09
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Capitol Planning Region, Connecticut","975328","09","110"],
["Greater Bridgeport Planning Region, Connecticut","327651","09","120"],
["Lower Connecticut River Valley Planning Region, Connecticut","176215","09","130"],
["Naugatuck Valley Planning Region, Co
--------------------------------------------------

STATE: 34
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Atlantic County, New Jersey","275213","34","001"],
["Bergen County, New Jersey","957736","34","003"],
["Burlington County, New Jersey","469167","34","005"],
["Camden County, New Jersey","527196","34","007"],
["Cape May County, New Jersey","94610","34","009
--------------------------------------------------

STATE: 36
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Albany County, New York","316659","36","001"],
["Bronx County, New York","1356476","36","005"],
["Broo

In [3]:
pop = pd.read_csv("co-est2025-alldata.csv", encoding = "latin1")

In [4]:
print(pop.columns.tolist())

['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'ESTIMATESBASE2020', 'POPESTIMATE2020', 'POPESTIMATE2021', 'POPESTIMATE2022', 'POPESTIMATE2023', 'POPESTIMATE2024', 'POPESTIMATE2025', 'NPOPCHG2020', 'NPOPCHG2021', 'NPOPCHG2022', 'NPOPCHG2023', 'NPOPCHG2024', 'NPOPCHG2025', 'BIRTHS2020', 'BIRTHS2021', 'BIRTHS2022', 'BIRTHS2023', 'BIRTHS2024', 'BIRTHS2025', 'DEATHS2020', 'DEATHS2021', 'DEATHS2022', 'DEATHS2023', 'DEATHS2024', 'DEATHS2025', 'NATURALCHG2020', 'NATURALCHG2021', 'NATURALCHG2022', 'NATURALCHG2023', 'NATURALCHG2024', 'NATURALCHG2025', 'INTERNATIONALMIG2020', 'INTERNATIONALMIG2021', 'INTERNATIONALMIG2022', 'INTERNATIONALMIG2023', 'INTERNATIONALMIG2024', 'INTERNATIONALMIG2025', 'DOMESTICMIG2020', 'DOMESTICMIG2021', 'DOMESTICMIG2022', 'DOMESTICMIG2023', 'DOMESTICMIG2024', 'DOMESTICMIG2025', 'NETMIG2020', 'NETMIG2021', 'NETMIG2022', 'NETMIG2023', 'NETMIG2024', 'NETMIG2025', 'RESIDUAL2020', 'RESIDUAL2021', 'RESIDUAL2022', 'RESIDUAL2023', 'RESIDUAL2024', 'RES

In [5]:
# Filtering for NY-NJ-CT
pop = pop[
    pop["STATE"].isin([9, 34, 36])
]

In [6]:
pop = pop[
    pop["COUNTY"] > 0
]

In [7]:
pop["fips"] = (
    pop["STATE"].astype(str).str.zfill(2)
    + pop["COUNTY"].astype(str).str.zfill(3)
)

In [8]:
pop = pop[[
    "fips",
    "STNAME",
    "CTYNAME",
    "POPESTIMATE2023"
]]

In [9]:
# Renaming columns
pop = pop.rename(columns={
    "STNAME": "state",
    "CTYNAME": "county",
    "POPESTIMATE2023": "population"
})

In [10]:
# Check data
print(pop.shape)
print(pop.head())

(92, 4)
      fips        state                                          county  \
316  09110  Connecticut                         Capitol Planning Region   
317  09120  Connecticut              Greater Bridgeport Planning Region   
318  09130  Connecticut  Lower Connecticut River Valley Planning Region   
319  09140  Connecticut                Naugatuck Valley Planning Region   
320  09150  Connecticut        Northeastern Connecticut Planning Region   

     population  
316      981775  
317      332081  
318      176419  
319      457384  
320       96790  


In [11]:
pop = pop[pop['state'] != 'Connecticut']

In [12]:
ct_towns = pd.read_csv("../../reference/ct_crosswalk/ct_town_crosswalk.csv")

In [13]:
# Load the population data
pop_df = pd.read_csv('../ct_towns_pop2023.csv')

# Standardize town names for merging (uppercase to match ct_towns)
pop_df['town_name'] = pop_df['town_name'].str.upper()

# Merge with ct_towns to get county info
merged = ct_towns[['town_name', 'county_name']].merge(
    pop_df,
    on='town_name',
    how='left'
)

# Group by county and sum populations
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)

print(county_pop)

         county_name  total_pop_2023
0   Fairfield County             0.0
1    Hartford County             0.0
2  Litchfield County             0.0
3   Middlesex County             0.0
4   New Haven County             0.0
5  New London County             0.0
6     Tolland County             0.0
7     Windham County             0.0


In [14]:
# Normalize both to uppercase for matching
ct_towns_copy = ct_towns.copy()
ct_towns_copy['town_name_upper'] = ct_towns_copy['town_name'].str.upper()
pop_df['town_name_upper'] = pop_df['town_name'].str.upper()

# Merge on the normalized column
merged = ct_towns_copy[['town_name', 'town_name_upper', 'county_name']].merge(
    pop_df[['town_name_upper', 'pop_2023']],
    on='town_name_upper',
    how='left'
)

# Check unmatched
unmatched = merged[merged['pop_2023'].isna()]['town_name'].tolist()
print("Unmatched towns:", unmatched)

# Group by county
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)
print(county_pop)

Unmatched towns: []
         county_name  total_pop_2023
0   Fairfield County          963780
1    Hartford County          898478
4   New Haven County          865717
5  New London County          268518
2  Litchfield County          186551
3   Middlesex County          166110
6     Tolland County          150906
7     Windham County          117116


In [15]:
# Get unique county_fips + county_name from ct_towns
county_fips_map = ct_towns[['county_fips', 'county_name']].drop_duplicates()

# Merge fips into county population df
ct_county_pop = county_pop.merge(
    county_fips_map,
    on='county_name',
    how='left'
)
ct_county_pop.set_index('county_fips', inplace = True)
print(ct_county_pop)

                   county_name  total_pop_2023
county_fips                                   
9001          Fairfield County          963780
9003           Hartford County          898478
9009          New Haven County          865717
9011         New London County          268518
9005         Litchfield County          186551
9007          Middlesex County          166110
9013            Tolland County          150906
9015            Windham County          117116


In [16]:
ct_county_pop['state'] = 'Connecticut'
pop['fips'] = pop['fips'].astype(int)
pop.set_index('fips', inplace = True)

In [17]:
ct_county_pop = ct_county_pop.rename(columns={'county_name': 'county', 'total_pop_2023':'population'})
ct_county_pop.index.name = 'fips'
ct_county_pop.head()

,county,population,state
fips,,,
9001,Fairfield County,963780,Connecticut
9003,Hartford County,898478,Connecticut
9009,New Haven County,865717,Connecticut
9011,New London County,268518,Connecticut
9005,Litchfield County,186551,Connecticut


In [18]:
pop_all = pd.concat([ct_county_pop, pop])

In [19]:
pop_all.head(20)

,county,population,state
fips,,,
9001,Fairfield County,963780,Connecticut
9003,Hartford County,898478,Connecticut
9009,New Haven County,865717,Connecticut
9011,New London County,268518,Connecticut
9005,Litchfield County,186551,Connecticut
9007,Middlesex County,166110,Connecticut
9013,Tolland County,150906,Connecticut
9015,Windham County,117116,Connecticut
34001,Atlantic County,276643,New Jersey


## Importing Stroke centers with fips

In [20]:
nj_stroke_centers = pd.read_csv("../geographic_accessibility_data/nj_all_stroke_centers_geocoded_with_fips.csv")
ny_stroke_centers = pd.read_csv("../geographic_accessibility_data/ny_all_stroke_centers_geocoded_with_fips.csv")

### FIlling in missing values for NJ

In [21]:
nj_stroke_centers.head(20)

,name,designation,address,latitude,longitude,fips
0,AtlanticCare Regional Medical Center,Comprehensive,"1925 Pacific Ave, Atlantic City, NJ 08401",39.357959,-74.433683,34001.0
1,Valley Hospital,Comprehensive,"223 N Van Dien Ave, Ridgewood, NJ 07450",40.982828,-74.101808,34003.0
2,Hackensack University Medical Center,Comprehensive,"30 Prospect Ave, Hackensack, NJ 07601",40.884467,-74.057638,34003.0
3,Cooper University Hospital,Comprehensive,"1 Cooper Plaza, Camden, NJ 08103",39.940854,-75.115774,34007.0
4,Our Lady of Lourdes Medical Center,Comprehensive,"1600 Haddon Ave, Camden, NJ 08103",NaN,NaN,NaN
5,University Hospital,Comprehensive,"150 Bergen St, Newark, NJ 07103",40.739774,-74.192687,34013.0
6,Saint Barnabas Medical Center,Comprehensive,"94 Old Short Hills Rd, Livingston, NJ 07039",40.765071,-74.301965,34013.0
7,Jefferson Washington Township Hospital,Comprehensive,"435 Hurffville-Cross Keys Rd, Turnersville, NJ...",39.733641,-75.064734,34015.0
8,Capital Health System at Fuld,Comprehensive,"750 Brunswick Ave, Trenton, NJ 08638",40.236022,-74.752690,34021.0
9,Robert Wood Johnson University Hospital New Br...,Comprehensive,"1 Robert Wood Johnson Pl, New Brunswick, NJ 08901",NaN,NaN,NaN


In [22]:
nan_rows = nj_stroke_centers[nj_stroke_centers['fips'].isna()]
print(nan_rows['name'].tolist())

['Our Lady of Lourdes Medical Center', 'Robert Wood Johnson University Hospital New Brunswick', 'Morristown Memorial Hospital', 'AtlanticCare Regional Medical Center - Jimmie Leeds Road', 'Englewood Hospital', 'Lourdes Medical Center Burlington County', 'East Orange General Hospital', 'Hackensack UMC - Mountainside', 'Capital Health Medical Center - Hopewell', 'Robert Wood Johnson University Hospital at Hamilton', 'Raritan Bay Medical Center Old Bridge']


In [23]:
# Placeholder dict — fill in with the correct FIPS codes for each facility
fips_lookup = {
    "Robert Wood Johnson University Hospital New Brunswick": 34019,
    "Morristown Memorial Hospital": 34027,
    "AtlanticCare Regional Medical Center - Jimmie Leeds Road": 34001,
    "Englewood Hospital": 34003,
    'Lourdes Medical Center Burlington County': 34007,
    'East Orange General Hospital' : 34013,
    "Hackensack UMC - Mountainside": 34013,  # Montclair → Essex County
    "Capital Health Medical Center - Hopewell": 34021,  # Pennington → Mercer County
    "Robert Wood Johnson University Hospital at Hamilton": 34021,  # Hamilton → Mercer County
    "Raritan Bay Medical Center Old Bridge": 34023,  # Old Bridge → Middlesex County
    'Our Lady of Lourdes Medical Center': 34007
}

# Get the rows with missing fips, so you can confirm names match exactly
nan_rows = nj_stroke_centers[nj_stroke_centers['fips'].isna()]
print(nan_rows['name'].tolist())

# Sanity check: make sure every NaN row's name is covered in the dict
missing_from_lookup = set(nan_rows['name']) - set(fips_lookup.keys())
assert not missing_from_lookup, f"These names aren't in fips_lookup: {missing_from_lookup}"

# Fill in using the name as the join key
nj_stroke_centers['fips'] = nj_stroke_centers['fips'].fillna(nj_stroke_centers['name'].map(fips_lookup))

['Our Lady of Lourdes Medical Center', 'Robert Wood Johnson University Hospital New Brunswick', 'Morristown Memorial Hospital', 'AtlanticCare Regional Medical Center - Jimmie Leeds Road', 'Englewood Hospital', 'Lourdes Medical Center Burlington County', 'East Orange General Hospital', 'Hackensack UMC - Mountainside', 'Capital Health Medical Center - Hopewell', 'Robert Wood Johnson University Hospital at Hamilton', 'Raritan Bay Medical Center Old Bridge']


In [24]:
# Check for missing fips values
missing_fips = nj_stroke_centers[nj_stroke_centers['fips'].isna()]

print(f"Number of rows with missing fips: {len(missing_fips)}")
print(missing_fips[['name', 'address']])

Number of rows with missing fips: 0
Empty DataFrame
Columns: [name, address]
Index: []


## Fill in missing values for NY

In [25]:
fips_lookup = {
    "Garnet Health Medical Center - Catskills Harris Campus": 36105,  # Harris → Sullivan County
    "Guthrie Corning Hospital": 36101,  # Corning → Steuben County
    "John T Mather Memorial Hospital": 36103,  # Port Jefferson → Suffolk County
    "Mercy Hospital": 36059,  # Rockville Centre → Nassau County
    "Mount Sinai South Nassau Hospital": 36059,  # Oceanside → Nassau County
}
nan_rows = ny_stroke_centers[ny_stroke_centers['fips'].isna()]
print(nan_rows['name'].tolist())

# Sanity check: make sure every NaN row's name is covered in the dict
missing_from_lookup = set(nan_rows['name']) - set(fips_lookup.keys())
assert not missing_from_lookup, f"These names aren't in fips_lookup: {missing_from_lookup}"

# Fill in using the name as the join key
ny_stroke_centers['fips'] = ny_stroke_centers['fips'].fillna(ny_stroke_centers['name'].map(fips_lookup))

['Garnet Health Medical Center - Catskills Harris Campus', 'Guthrie Corning Hospital', 'John T Mather Memorial Hospital', 'Mercy Hospital', 'Mount Sinai South Nassau Hospital']


In [26]:
# Check for missing fips values
missing_fips = nj_stroke_centers[nj_stroke_centers['fips'].isna()]

print(f"Number of rows with missing fips: {len(missing_fips)}")
print(missing_fips[['name', 'address']])

Number of rows with missing fips: 0
Empty DataFrame
Columns: [name, address]
Index: []


## Importing CT

In [27]:
ct_advanced = pd.read_csv("../geographic_accessibility_data/ct_advanced_geocoded_with_fips.csv")
ct_basic = pd.read_csv("../geographic_accessibility_data/ct_basic_geocoded_with_fips.csv")

In [28]:
ct_basic.head()

,name,group,latitude,longitude,fips
0,Bridgeport Hospital Milford Campus,Basic,41.216545,-73.065360,9009
1,Charlotte Hungerford Hospital,Basic,41.792271,-73.133769,9005
2,Day Kimball Hospital,Basic,41.906093,-71.913028,9015
3,Greenwich Hospital,Basic,41.034299,-73.630646,9001
4,Griffin Hospital,Basic,41.336435,-73.090512,9009


In [29]:
ct_advanced.head()

,name,group,latitude,longitude,fips
0,Bridgeport Hospital,Advanced,41.189471,-73.167470,9001
1,Danbury Hospital,Advanced,41.405218,-73.445159,9001
2,Hartford Hospital,Advanced,41.754069,-72.679390,9003
3,Norwalk Hospital,Advanced,41.110829,-73.421953,9001
4,Saint Francis Hospital and Medical Center,Advanced,41.774141,-72.698084,9003


In [30]:
ct_stroke_centers = pd.concat([ct_advanced, ct_basic])


In [31]:
ct_stroke_centers.head()

,name,group,latitude,longitude,fips
0,Bridgeport Hospital,Advanced,41.189471,-73.167470,9001
1,Danbury Hospital,Advanced,41.405218,-73.445159,9001
2,Hartford Hospital,Advanced,41.754069,-72.679390,9003
3,Norwalk Hospital,Advanced,41.110829,-73.421953,9001
4,Saint Francis Hospital and Medical Center,Advanced,41.774141,-72.698084,9003


In [32]:
ct_stroke_centers['state'] = 'Connecticut'
ct_stroke_centers.drop(columns=['latitude', 'longitude'], inplace = True) 
ct_stroke_centers.head()

,name,group,fips,state
0,Bridgeport Hospital,Advanced,9001,Connecticut
1,Danbury Hospital,Advanced,9001,Connecticut
2,Hartford Hospital,Advanced,9003,Connecticut
3,Norwalk Hospital,Advanced,9001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,9003,Connecticut


In [33]:
ct_stroke_centers.rename(columns={"group": "designation"}, inplace=True)

In [34]:
ct_stroke_centers['fips'] = ct_stroke_centers['fips'].astype(str).str.zfill(5)
ct_stroke_centers

,name,designation,fips,state
0,Bridgeport Hospital,Advanced,09001,Connecticut
1,Danbury Hospital,Advanced,09001,Connecticut
2,Hartford Hospital,Advanced,09003,Connecticut
3,Norwalk Hospital,Advanced,09001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,09003,Connecticut
5,St. Vincent's Medical Center,Advanced,09001,Connecticut
6,Yale New Haven Hospital,Advanced,09009,Connecticut
0,Bridgeport Hospital Milford Campus,Basic,09009,Connecticut
1,Charlotte Hungerford Hospital,Basic,09005,Connecticut
2,Day Kimball Hospital,Basic,09015,Connecticut


### Cleaning and Merging

In [35]:
nj_stroke_centers.drop(columns=['latitude', 'longitude'], inplace = True)
ny_stroke_centers.drop(columns=['latitude', 'longitude'], inplace = True)

In [36]:
nj_stroke_centers.drop(columns=['address'], inplace = True)
ny_stroke_centers.drop(columns=['address'], inplace = True)

In [37]:
nj_stroke_centers.head()

,name,designation,fips
0,AtlanticCare Regional Medical Center,Comprehensive,34001.0
1,Valley Hospital,Comprehensive,34003.0
2,Hackensack University Medical Center,Comprehensive,34003.0
3,Cooper University Hospital,Comprehensive,34007.0
4,Our Lady of Lourdes Medical Center,Comprehensive,34007.0


In [38]:
nj_stroke_centers['state'] = 'New Jersey'
ny_stroke_centers['state'] = 'New York'

In [39]:
stroke_centers = pd.concat([ct_stroke_centers, nj_stroke_centers, ny_stroke_centers])
stroke_centers

,name,designation,fips,state
0,Bridgeport Hospital,Advanced,09001,Connecticut
1,Danbury Hospital,Advanced,09001,Connecticut
2,Hartford Hospital,Advanced,09003,Connecticut
3,Norwalk Hospital,Advanced,09001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,09003,Connecticut
...,...,...,...,...
117,Syosset Hospital,Primary Stroke Center,36059.0,New York
118,Unity Hospital of Rochester,Primary Stroke Center,36055.0,New York
119,United Memorial Medical Center North Street Ca...,Primary Stroke Center,36037.0,New York
120,University Hospital of Brooklyn (SUNY Downstate),Primary Stroke Center,36047.0,New York


In [40]:
stroke_centers['fips'] = stroke_centers['fips'].astype(float).astype(int).astype(str).str.zfill(5)
stroke_centers

,name,designation,fips,state
0,Bridgeport Hospital,Advanced,09001,Connecticut
1,Danbury Hospital,Advanced,09001,Connecticut
2,Hartford Hospital,Advanced,09003,Connecticut
3,Norwalk Hospital,Advanced,09001,Connecticut
4,Saint Francis Hospital and Medical Center,Advanced,09003,Connecticut
...,...,...,...,...
117,Syosset Hospital,Primary Stroke Center,36059,New York
118,Unity Hospital of Rochester,Primary Stroke Center,36055,New York
119,United Memorial Medical Center North Street Ca...,Primary Stroke Center,36037,New York
120,University Hospital of Brooklyn (SUNY Downstate),Primary Stroke Center,36047,New York


In [41]:
#matching fips to county
fips_to_county = pd.read_csv('../ny_nj_ct_fips.csv')

In [42]:
fips_to_county

,fips,county,state
0,36001,Albany,NY
1,36003,Allegany,NY
2,36005,Bronx,NY
3,36007,Broome,NY
4,36009,Cattaraugus,NY
...,...,...,...
86,9007,Middlesex,CT
87,9009,New Haven,CT
88,9011,New London,CT
89,9013,Tolland,CT


In [43]:
fips_to_county['fips'] = fips_to_county['fips'].astype(str).str.zfill(5)
fips_to_county

,fips,county,state
0,36001,Albany,NY
1,36003,Allegany,NY
2,36005,Bronx,NY
3,36007,Broome,NY
4,36009,Cattaraugus,NY
...,...,...,...
86,09007,Middlesex,CT
87,09009,New Haven,CT
88,09011,New London,CT
89,09013,Tolland,CT


In [44]:
# Merge to bring in the county column
stroke_centers = stroke_centers.merge(
    fips_to_county[['fips', 'county']],
    on='fips',
    how='left'
)

# Check for any unmatched rows
unmatched = stroke_centers[stroke_centers['county'].isna()]
print(f"Unmatched rows: {len(unmatched)}")
if len(unmatched) > 0:
    print(unmatched[['name', 'fips', 'state']])


Unmatched rows: 0


In [45]:
stroke_centers = stroke_centers.drop(columns = ['designation'])

In [46]:
stroke_centers

,name,fips,state,county
0,Bridgeport Hospital,09001,Connecticut,Fairfield
1,Danbury Hospital,09001,Connecticut,Fairfield
2,Hartford Hospital,09003,Connecticut,Hartford
3,Norwalk Hospital,09001,Connecticut,Fairfield
4,Saint Francis Hospital and Medical Center,09003,Connecticut,Hartford
...,...,...,...,...
213,Syosset Hospital,36059,New York,Nassau
214,Unity Hospital of Rochester,36055,New York,Monroe
215,United Memorial Medical Center North Street Ca...,36037,New York,Genesee
216,University Hospital of Brooklyn (SUNY Downstate),36047,New York,Kings


In [47]:
# Zero-pad pop_all fips index
pop_all.index = pop_all.index.astype(str).str.zfill(5)

# Count stroke centers per fips
stroke_counts = stroke_centers.groupby('fips').size().rename('stroke_center_count')

# Build the final dataframe: one row per fips with county and population
fips_info = stroke_centers[['fips', 'county']].drop_duplicates().set_index('fips')

# Combine
final = fips_info.join(stroke_counts)
final['population'] = pop_all['population']
final['stroke_centers_per_100k'] = (final['stroke_center_count'] / final['population']) * 100_000

print(final)

           county  stroke_center_count  population  stroke_centers_per_100k
fips                                                                       
09001   Fairfield                    6      963780                 0.622549
09003    Hartford                    6      898478                 0.667796
09009   New Haven                    7      865717                 0.808578
09005  Litchfield                    3      186551                 1.608139
09015     Windham                    2      117116                 1.707709
...           ...                  ...         ...                      ...
36079      Putnam                    1       98110                 1.019264
36083  Rensselaer                    1      159605                 0.626547
36045   Jefferson                    1      113280                 0.882768
36091    Saratoga                    1      238699                 0.418938
36037     Genesee                    1       58016                 1.723662

[69 rows x 

In [48]:
final

,county,stroke_center_count,population,stroke_centers_per_100k
fips,,,,
09001,Fairfield,6,963780,0.622549
09003,Hartford,6,898478,0.667796
09009,New Haven,7,865717,0.808578
09005,Litchfield,3,186551,1.608139
09015,Windham,2,117116,1.707709
...,...,...,...,...
36079,Putnam,1,98110,1.019264
36083,Rensselaer,1,159605,0.626547
36045,Jefferson,1,113280,0.882768


In [49]:
print(final[final['stroke_centers_per_100k'].isna()])

Empty DataFrame
Columns: [county, stroke_center_count, population, stroke_centers_per_100k]
Index: []


In [50]:
# Zero-pad pop_all fips index
pop_all.index = pop_all.index.astype(str).str.zfill(5)

# Count stroke centers per fips
stroke_counts = stroke_centers.groupby('fips').size().rename('stroke_center_count')

# Start from pop_all so all 91 counties are included
final = pop_all[['county', 'population', 'state']].copy()

# Join counts (counties with no centers will get NaN, fill with 0)
final = final.join(stroke_counts)
final['stroke_center_count'] = final['stroke_center_count'].fillna(0).astype(int)

# Calculate rate
final['stroke_centers_per_100k'] = (final['stroke_center_count'] / final['population']) * 100_000

print(final.shape)
print(final)

(91, 5)
                   county  population        state  stroke_center_count  \
fips                                                                      
09001    Fairfield County      963780  Connecticut                    6   
09003     Hartford County      898478  Connecticut                    6   
09009    New Haven County      865717  Connecticut                    7   
09011   New London County      268518  Connecticut                    2   
09005   Litchfield County      186551  Connecticut                    3   
...                   ...         ...          ...                  ...   
36115   Washington County       60052     New York                    0   
36117        Wayne County       90721     New York                    1   
36119  Westchester County      998335     New York                   10   
36121      Wyoming County       39750     New York                    0   
36123        Yates County       24367     New York                    0   

       stroke_ce

In [51]:
final

,county,population,state,stroke_center_count,stroke_centers_per_100k
fips,,,,,
09001,Fairfield County,963780,Connecticut,6,0.622549
09003,Hartford County,898478,Connecticut,6,0.667796
09009,New Haven County,865717,Connecticut,7,0.808578
09011,New London County,268518,Connecticut,2,0.744829
09005,Litchfield County,186551,Connecticut,3,1.608139
...,...,...,...,...,...
36115,Washington County,60052,New York,0,0.000000
36117,Wayne County,90721,New York,1,1.102281
36119,Westchester County,998335,New York,10,1.001668


## Checking and Saving to File

In [52]:
print(final['stroke_center_count'].sum())

218


In [53]:
print(final.isna().sum())

county                     0
population                 0
state                      0
stroke_center_count        0
stroke_centers_per_100k    0
dtype: int64


In [54]:
final['county'] = final['county'].str.replace(' County', '', regex=False)
print(final['county'].head(10))

fips
09001     Fairfield
09003      Hartford
09009     New Haven
09011    New London
09005    Litchfield
09007     Middlesex
09013       Tolland
09015       Windham
34001      Atlantic
34003        Bergen
Name: county, dtype: object


In [55]:
#final.to_csv('stroke_centers_per_100k.csv')

## Neurologists per 100k

### Reading AHRF data

In [56]:
ahrf = pd.read_csv(
    "ahrf2023.csv",
    encoding="latin1",
    usecols=[
        "fips_st_cnty",
        "st_name_abbrev",
        "cnty_name",
        "md_nf_neuro_21"
    ],
    low_memory=False
)

In [ ]:
ah

Next, we select only NY, NJ, and CT.

In [69]:
ahrf = ahrf[
    ahrf["st_name_abbrev"].isin(["NY", "NJ", "CT"])
].copy()

We rename columns to keep them consistent with our database:

In [71]:
ahrf = ahrf.rename(columns={
    "fips_st_cnty": "fips",
    "cnty_name": "county",
    "st_name_abbrev": "state",
    "md_nf_neuro_21": "neurologists"
})

We also make sure that fips codes are 5-digit strings.

In [73]:
ahrf["fips"] = (
    ahrf["fips"]
    .astype(str)
    .str.zfill(5)
)

We merge this with population data so that we can calculate per 100,000 rate.

In [75]:
# Making sure fips codes are strings for population data

pop_all.index = (
    pop_all.index
    .astype(str)
    .str.zfill(5)
)

In [76]:
# Adding population column to neurologist dataset

neurology = ahrf.merge(
    pop_all[["population"]],
    left_on="fips",
    right_index=True,
    how="left"
)

In [77]:
neurology.head()

,fips,state,county,neurologists,population
313,09001,CT,Fairfield,58.0,963780
314,09003,CT,Hartford,61.0,898478
315,09005,CT,Litchfield,7.0,186551
316,09007,CT,Middlesex,8.0,166110
317,09009,CT,New Haven,141.0,865717


Next, we calculate neurologists per 100k:

In [79]:
neurology["neurologists_per_100k"] = (
    neurology["neurologists"] /
    neurology["population"]
) * 100000

#### Saving to csv file:

In [81]:
#neurology.to_csv(
    "neurologists_per_100k.csv",
    index=False
)

## Hospital Beds per 100k

### Reading CMS Hospital Cost Report

In [84]:
# Reading full CMS 2023 dataset

hosp = pd.read_csv(
    "CostReport_2023_Final.csv",
    low_memory=False
)

### Cleaning CMS Hospital Cost Report Data

#### Obtaining FIPS Codes

First, we need to standardize hospital county names in the CMS Hospital Cost Report Data.

In [88]:
hosp["County"] = (
    hosp["County"]
    .str.upper()
    .str.replace(" COUNTY", "", regex=False)
    .str.strip()
)

hosp["State Code"] = (
    hosp["State Code"]
    .str.upper()
    .str.strip()
)

We will also rename columns to be consistent with our database.

In [90]:
hosp = hosp.rename(columns={
    "County": "county",
    "State Code": "state"
})

Next, we must standardize the crosswalk data:

In [92]:
crosswalk = pd.read_csv("../ny_nj_ct_fips.csv")

crosswalk["county"] = (
    crosswalk["county"]
    .str.upper()
    .str.strip()
)

crosswalk["state"] = (
    crosswalk["state"]
    .str.upper()
    .str.strip()
)

crosswalk["fips"] = (
    crosswalk["fips"]
    .astype(str)
    .str.zfill(5)
)

We can merge these two files to get the FIPS codes:

In [94]:
hosp = hosp.merge(
    crosswalk,
    left_on=["county", "state"],
    right_on=["county", "state"],
    how="left"
)

In [95]:
# Keeping only NY, NJ, and CT hospitals that matched the crosswalk

hosp = hosp[hosp["fips"].notna()].copy()

# Making sure FIPS are strings

hosp["fips"] = (
    hosp["fips"]
    .astype(str)
    .str.zfill(5)
)

We will only keep the variables that we need:

In [97]:
hosp = hosp[
    [
        "Provider Type",
        "Hospital Name",
        "state",
        "county",
        "Number of Beds",
        "fips"
    ]
]

We will only keep general short-term hospitals.

In [99]:
hosp = hosp[hosp["Provider Type"] == 1]

Next, we filter the data for NY-NJ-CT.

In [101]:
hosp = hosp[
    hosp["state"].isin(["NY", "NJ", "CT"])
]

We also must convert number of beds to numeric:

In [103]:
hosp["Number of Beds"] = pd.to_numeric(
    hosp["Number of Beds"],
    errors="coerce"
)

In [104]:
hosp.head()

,Provider Type,Hospital Name,state,county,Number of Beds,fips
26,1,EASTERN NIAGARA HOSPITAL,NY,NIAGARA,109.0,36063
60,1,THE GRIFFIN HOSPITAL,CT,NEW HAVEN,101.0,09009
107,1,WINDHAM COMMUNITY MEMORIAL HOSPITAL,CT,WINDHAM,46.0,09015
143,1,ELIZABETHTOWN COMMUNITY HOSPITAL,NY,ESSEX,25.0,36031
232,1,CHARLOTTE HUNGERFORD HOSPITAL,CT,LITCHFIELD,108.0,09005


### Computing Hospital Beds per 100k

#### Summing Beds Within Each County

In [107]:
county_beds = (
    hosp
    .groupby(["fips", "county", "state"], as_index=False)
    ["Number of Beds"]
    .sum()
    .rename(columns={"Number of Beds": "hospital_beds"})
)

county_beds = (
    pop_all[["county", "state", "population"]]
    .merge(
        county_beds,
        left_index=True,
        right_on="fips",
        how="left"
    )
)

county_beds["hospital_beds"] = county_beds["hospital_beds"].fillna(0)
county_beds["hospital_beds_per_100k"] = (
    county_beds["hospital_beds"] /
    county_beds["population"]
) * 100000

In [108]:
county_beds.head(10)

,county_x,state_x,population,fips,county_y,state_y,hospital_beds,hospital_beds_per_100k
0.0,Fairfield County,Connecticut,963780,09001,FAIRFIELD,CT,1606.0,166.635539
1.0,Hartford County,Connecticut,898478,09003,HARTFORD,CT,1795.0,199.782299
4.0,New Haven County,Connecticut,865717,09009,NEW HAVEN,CT,1825.0,210.807920
5.0,New London County,Connecticut,268518,09011,NEW LONDON,CT,423.0,157.531339
2.0,Litchfield County,Connecticut,186551,09005,LITCHFIELD,CT,158.0,84.695338
3.0,Middlesex County,Connecticut,166110,09007,MIDDLESEX,CT,187.0,112.576004
6.0,Tolland County,Connecticut,150906,09013,TOLLAND,CT,146.0,96.748970
7.0,Windham County,Connecticut,117116,09015,WINDHAM,CT,150.0,128.078145
8.0,Atlantic County,New Jersey,276643,34001,ATLANTIC,NJ,731.0,264.239471
9.0,Bergen County,New Jersey,964156,34003,BERGEN,NJ,1902.0,197.270981


#### Merging with Population Data

In [110]:
pop_all.index = (
    pop_all.index
    .astype(str)
    .str.zfill(5)
)

county_beds = county_beds.merge(
    pop_all[["population"]],
    left_on="fips",
    right_index=True,
    how="left"
)

In [111]:
# Renaming and dropping duplicates

county_beds = county_beds.rename(columns={
    "county_x": "county",
    "state_x": "state",
    "population_x": "population"
})

county_beds = county_beds.drop(columns=[
    "county_y",
    "state_y",
    "population_y"
])

In [112]:
county_beds.head()

,county,state,population,fips,hospital_beds,hospital_beds_per_100k
0.0,Fairfield County,Connecticut,963780,09001,1606.0,166.635539
1.0,Hartford County,Connecticut,898478,09003,1795.0,199.782299
4.0,New Haven County,Connecticut,865717,09009,1825.0,210.807920
5.0,New London County,Connecticut,268518,09011,423.0,157.531339
2.0,Litchfield County,Connecticut,186551,09005,158.0,84.695338


#### Calculating Hospital Beds per 100k 

In [114]:
county_beds["hospital_beds_per_100k"] = (
    county_beds["hospital_beds"] /
    county_beds["population"]
) * 100000

### Saving as a CSV

In [116]:
#county_beds.to_csv(
    "hospital_beds_per_100k.csv",
    index=False
)